# Survey data analysis

## 1. Extract and read data

We are expecting a zip file containing all files related to the survey to be exported from the survey platform (e.g. Typeform). In that zipfile, there should be a `.csv` file containing the survey responses data. Make sure to only use data with no [personally identifiable information](https://en.wikipedia.org/wiki/Personal_data).

In [2]:
import zipfile
import pathlib
import pandas as pd

in_path = "/home/melissa/projects/napari-survey-data/no-pii"
out_path = "_data"

past_surveys = {
    "2020": "napari-survey-2020-partial-export.csv",
    "2021": "napari-survey-2021-partial-export.csv",
    "2022": "napari-survey-2022-partial-export.csv",
    "2023": "responses copy - agreed - no PII.csv",
}

data = []
for year, csv_filename in past_surveys.items():
    zip_filename = f"{year}.zip"
    with zipfile.ZipFile(pathlib.Path(in_path, zip_filename), 'r') as zfile:
        zfile.extract(csv_filename, out_path)

    data.append(pd.read_csv(pathlib.Path(out_path, csv_filename)))

## 2. Prepare data

### 2023

In [5]:
data_df = data[-1]
# Clean up unnecessary columns
data_df.drop(data_df.columns[[0, 1]], axis=1, inplace=True)
data_df.drop(list(data_df)[95:102], axis=1, inplace=True)

In [6]:
data_df

,Academia,Government,Non-profit,Industry,None,Other,Principal investigator,Postdoctoral researcher,Student,(Bio)image/data analyst,...,Improved access to features from the GUI and better documentation,Improved interactivity and performance when visualizing data,"Layer improvements (slicing performance, layer groups, consistency across layer types)",Improved opening and saving of data,"Improved sharing of data between the viewer and plugins, or between different plugins",Bug fixes,Other.6,"Overall, how satisfied or dissatisfied are you with napari?",Feel free to elaborate here on your rating in the previous question.,"Please share any additional thoughts, feedback, or suggestions to improve napari or napari hub."
0,Academia,NaN,NaN,NaN,NaN,NaN,Principal investigator,NaN,NaN,(Bio)image/data analyst,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,NaN
1,Academia,NaN,NaN,NaN,NaN,NaN,Principal investigator,NaN,NaN,(Bio)image/data analyst,...,NaN,NaN,NaN,NaN,Improved sharing of data between the viewer an...,Bug fixes,Add a histogram next to the brightness/ contra...,4,NaN,When will the conda-based installer of napari+...
2,Academia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Student,NaN,...,NaN,NaN,NaN,Improved opening and saving of data,NaN,NaN,NaN,5,NaN,"It would be really, really helpful if we could..."
3,Academia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Improved opening and saving of data,Improved sharing of data between the viewer an...,Bug fixes,NaN,5,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,Small LLC,NaN,NaN,NaN,(Bio)image/data analyst,...,NaN,NaN,NaN,NaN,NaN,NaN,Improved handling of meta data and large compl...,4,NaN,Would love to have real life development meeti...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,Academia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Student,NaN,...,Improved access to features from the GUI and b...,NaN,"Layer improvements (slicing performance, layer...",NaN,NaN,Bug fixes,NaN,5,NaN,NaN
119,NaN,NaN,Non-profit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,NaN,NaN
120,Academia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,(Bio)image/data analyst,...,NaN,Improved interactivity and performance when vi...,NaN,NaN,Improved sharing of data between the viewer an...,NaN,NaN,4,NaN,NaN
121,Academia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Student,(Bio)image/data analyst,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,Thank you for all your hard work


### Filter multiple choice data

Multiple choice questions are organized as multiple columns corresponding to each of the options available as answers.

For each of the answer columns, the contents are either NaN or the selected choice (i.e. same as the column name).

In [ ]:
questions_map = {
    "Which kind of organization(s) are you associated with?": (0,6),
    "Which of the following describe your current role?": (6, 14),
    "Which domains / industries do you work in?": (14,),
    "Which of these best describes your Python programming experience level?": (15,),
    "Approximately how long have you used napari?": (16, 22),
    "You indicated that you haven't used napari, or no longer use it. Please share why.": (22,),
    "How have you used napari?": (23, 30),
    "Which types of images have you worked with during the past year, or plan to work with in upcoming projects?": (30, 41),
    "In the context of image visualization, annotation, and analysis, what tools do you use other than napari, and why?": (41,),
    # Indicate the degree to which you agree or disagree with the following statements in the context of *your* napari usage and/or contributions.
    # Strongly disagree, Somewhat disagree, Neutral, Somewhat agree, Strongly agree, Unfamiliar/not applicable
    "There are adequate napari tutorials": (42,),
    "There is adequate napari API documentation": (43,),
    "The napari community is welcoming and inclusive": (44,),
    "It is easy to contribute code to the napari project": (45,),
    "It is easy to contribute documentation to the napari project": (46,),
    "It is easy to develop plugins for napari": (47,),
    "Feel free to elaborate here on your ratings in the previous question.": (48,),
    # Rate napari viewer on the following aspects.
    # Poor, Fair, Good, Very Good, Excellent, NA/Not sure
    "Ease of installation": (49,),
    "Conflicts due to software dependencies": (50,),
    "Start time": (51,),
    "Performance viewing data": (52,),
    "Existing feature set excluding plugins": (53,),
    "Pace of new feature development": (54,),
    "Introduction of breaking API changes": (55,),
    "Feel free to elaborate here on your ratings in the previous question.": (56,),
    # Indicate the ways in which you have engaged in the [napari community](https://napari.org/stable/community/index.html).
    # Unfamiliar, Aware, Reader, Participant
    "GitHub issues/discussions": (57,),
    "GitHub code/doc. submissions/NAPs/reviews": (58,),
    "Image.sc forum": (59,),
    "napari Zulip chat": (60,),
    "Mastodon": (61,),
    "X (Twitter)": (62,),
    "Conferences": (63,),
    "Community meetings": (64,),
    "How satisfied or dissatisfied are you with your *experience using napari with plugins*?": (65,),
    "Feel free to elaborate here on your choice in the previous question.": (66,),
    # Which sources do you use to find napari plugins?
    "I ask a colleague": (67,),
    "GitHub": (68,),
    "Image.sc forum": (69,),
    "napari hub": (70,),
    "napari plugin installation window": (71,),
    "Mastodon": (72,),
    "X (Twitter)": (73,),
    "napari Zulip chat": (74,),
    "Web search": (75,),
    "Other": (76,),
    "How often are you able to find a napari plugin that meets your needs?": (77,),
    # The napari team is considering opt-in, anonymous data gathering to better understand usage of napari, along with a public dashboard of aggregated data. The corresponding _napari Advancement Proposal_ on telemetry, [NAP-8](https://napari.org/dev/naps/8-telemetry.html) is open to discussion on Zulip chat, Image.sc forum, and GitHub, and you are welcome to participate. In the next two questions, we aim to better understand where the community stands, including users who may not be active in the discussion avenues. The proposal may or may not move forward depending on community feedback
    # The level of data collection is proposed to be adjustable:\n• Basic: includes software and hardware configuration information linked to an identifier updated weekly to prevent tracking any individual user.\n• Middle: basic level + names and versions of public plugins installed in stable versions of napari\n• Full: middle level + usage of plugin features/functions and corresponding [contributions](https://napari.org/stable/plugins/contributions.html), and type and size of image data
    "If a future version of napari were to introduce opt-in, anonymous data gathering, what would be your preferred course of action?": (78,),
    "Other": (79,),
    "What questions or concerns might you have about the introduction of opt-in, anonymous data gathering in a future version of napari (along with a public dashboard of the aggregated data on napari.org)?": (80,),
    # Select up to three areas of improvement from the following list that would be most valuable to you. We will use this information to help us prioritize the napari roadmap.",
    "Access napari in a Jupyter notebook/JupyterLab/other web-based UI": (81,),
    "View image data without the whole UI, and related API improvements to improve consistency and allow reuse of viewer components": (82,),
    "Better tools to annotate (e.g. manually segment) 3D data": (83,),
    "Multiple canvases, such as for orthogonal views, synced views, or simultaneous 2D/3D rendering": (84,),
    "Improved access to features from the GUI and better documentation": (85,),
    "Improved interactivity and performance when visualizing data": (86,),
    "Layer improvements (slicing performance, layer groups, consistency across layer types)": (87,),
    "Improved opening and saving of data": (88,),
    "Improved sharing of data between the viewer and plugins, or between different plugins": (89,),
    "Bug fixes": (90,),
    "Other": (91,),
    "Overall, how satisfied or dissatisfied are you with napari?": (92,),
    "Feel free to elaborate here on your rating in the previous question.": (93,),
    "Please share any additional thoughts, feedback, or suggestions to improve napari or napari hub.": (94,),
}

## Plots

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
title = "Which of the following describe your current role?"
data_df[data_df.columns[6:14]]

In [ ]:
y = []
yticks = []
for col in range(6,14):
    y.append(data_df[data_df.columns[col]].count())
    yticks.append(data_df.columns[col])

# plot
fig, ax = plt.subplots()
ax.set_title(title)
ax.barh(range(6,14), y, edgecolor="white", linewidth=0.5, tick_label=yticks)

---

In [ ]:
for index, col in enumerate(data_df.columns):
    print(f"{index}: {col}")

In [ ]:
data_df[data_df.columns[49]]

In [ ]:
data_df[data_df.columns[0:6]].groupby(

In [ ]:
data_df[data_df.columns[0:6]].pivot(index=data_df.index, columns=)

In [ ]:
animals = pd.DataFrame(
    {
        "kind": ["cat", "dog", "cat", "dog"],
        "height": [9.1, 6.0, 9.5, 34.0],
        "weight": [7.9, 7.5, 9.9, 198.0],
    }
)

In [ ]:
animals